# Лабораторная работа 16. Механистическая интерпретируемость: logit lens, induction heads, activation patching (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 6:** интерпретируемость и in-context learning

**Цель.** Научиться находить механизмы внутри обученного трансформера с помощью ручных хуков (без TransformerLens): читать residual stream, находить induction heads и локализовать цепь для задачи IOI.

**Что нужно сдать:**
1. Logit lens для GPT-2 small: как предсказание формируется по слоям.
2. Induction heads: оценки для всех 144 голов, тепловая карта, потери на повторённой последовательности.
3. Activation patching на задаче Indirect Object Identification (IOI): по слоям и позициям residual stream и по выходам голов.
4. Абляцию найденных голов в сравнении со случайными.
5. Дополнительно: появление induction heads по чекпойнтам Pythia.

**Теория.** Residual stream — сумма вкладов эмбеддинга, всех голов и MLP: \(x_L = x_0 + \sum_{l}\big(\sum_h \mathrm{head}_{l,h}(x) + \mathrm{MLP}_l(x)\big)\). Выход головы: \(\mathrm{head}_{l,h} = z_{l,h} W_O^{(h)}\), где \(z_{l,h} = A_{l,h} V\). В GPT-2 \(z\) всех голов конкатенируется и подаётся в `attn.c_proj`, поэтому pre-hook этого слоя даёт доступ к \(z\) каждой головы (срез из 64 координат).

**Induction head** реализует правило \([A][B] \dots [A] \to [B]\): из позиции второго \(A\) внимание идёт на токен, следующий за первым \(A\). Обычно работает в паре с предыдущей «previous token head» (Olsson et al., 2022).

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("gpt2")
try: model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(DEVICE).eval()
except TypeError: model = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)
L, H = model.config.n_layer, model.config.n_head; DH = model.config.n_embd // H
print(f"GPT-2 small: {L} слоёв × {H} голов, d_head = {DH}")

## 1. Logit lens

Примените финальный LayerNorm и unembedding к residual stream после каждого слоя (`output_hidden_states=True`).
1. Для фразы выведите топ-1 токен каждого слоя в каждой позиции.
2. На отрывке текста (около 1000 токенов) посчитайте по слоям среднюю log-вероятность правильного следующего токена и долю позиций, где он топ-1.

In [ ]:
def logit_lens(ids):
    """Возвращает логиты (L+1, B, T, V) для residual stream после эмбеддинга и каждого блока."""
    # TODO: ваш код
    raise NotImplementedError

phrase = "The Eiffel Tower is located in the city of"
ids = tok(phrase, return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): ll = logit_lens(ids)
toks = [tok.decode(t) for t in ids[0]]
print("слой | " + " | ".join(f"{t!r:>10}" for t in toks[-5:]))
for l in range(0, L + 1, 2):
    print(f"{l:4d} | " + " | ".join(f"{tok.decode(ll[l, 0, t].argmax())!r:>10}" for t in range(len(toks) - 5, len(toks))))

In [ ]:
text = ("The history of the printing press begins in the fifteenth century, when Johannes Gutenberg developed a system of movable "
        "metal type in the German city of Mainz. Before this invention, books were copied by hand, a slow and expensive process "
        "that limited the spread of knowledge. Within a few decades printing shops appeared in many European cities, and the "
        "number of books in circulation grew from thousands to millions. The new technology changed religion, science and "
        "politics: ideas could now travel faster than ever before, and scholars in different countries could read the same texts. ") * 4
ids = tok(text, return_tensors="pt").input_ids[:, :1000].to(DEVICE)
# TODO: ваш код (по слоям — средняя log p правильного токена и top-1 accuracy, график)

## 2. Induction heads

Последовательность из 50 случайных токенов, повторённая дважды (с BOS в начале). Для каждой головы induction score — средний вес внимания из позиции \(t\) второй половины на позицию \(t - 50 + 1\) (токен после предыдущего вхождения). Также посчитайте previous-token score (внимание на \(t-1\)) и сравните loss на первой и второй половинах.

In [ ]:
SEQ, NB = 50, 8
set_seed(0)
rand = torch.randint(1000, 20000, (NB, SEQ))
rep = torch.cat([torch.full((NB, 1), tok.bos_token_id), rand, rand], 1).to(DEVICE)

# TODO: ваш код (attentions → induction и previous-token scores (L×H), loss по половинам)

## 3. IOI и activation patching

Промпт: «When Mary and John went to the store, John gave a drink to» → « Mary». Метрика — разность логитов \(\mathrm{LD} = \text{logit}(IO) - \text{logit}(S)\). Испорченный промпт: второе упоминание S заменено третьим именем C, и подсказка о том, кто «повторился», пропадает.

Нормированное восстановление: \(\frac{\mathrm{LD}_{patched} - \mathrm{LD}_{corr}}{\mathrm{LD}_{clean} - \mathrm{LD}_{corr}}\).

1. Patching residual stream: для каждого слоя \(l\) и позиции \(p\) подставьте в испорченный прогон выход блока \(l\) на позиции \(p\) из чистого прогона.
2. Patching голов: подставьте \(z_{l,h}\) из чистого прогона (на всех позициях) для каждой из 144 голов.

In [ ]:
NAMES = [n for n in ["Mary", "John", "Tom", "James", "Paul", "Mark", "Sarah", "Anna", "Lisa", "David", "Kevin", "Laura", "Peter", "Emma", "Jack"]
         if len(tok.encode(" " + n)) == 1]
TEMPLATES = ["When {A} and {B} went to the store, {C} gave a drink to",
             "When {A} and {B} got a snack at the cafe, {C} decided to give it to",
             "After {A} and {B} went to the park, {C} handed a ball to"]
rng = random.Random(0)
def ioi_batch(template, n):
    clean, corr, io, s = [], [], [], []
    for _ in range(n):
        a, b, c = rng.sample(NAMES, 3)
        if rng.random() < 0.5: A, B = a, b
        else: A, B = b, a
        clean.append(template.format(A=A, B=B, C=b)); corr.append(template.format(A=A, B=B, C=c))
        io.append(tok.encode(" " + a)[0]); s.append(tok.encode(" " + b)[0])
    enc = lambda xs: torch.tensor([tok.encode(x) for x in xs]).to(DEVICE)
    return enc(clean), enc(corr), torch.tensor(io).to(DEVICE), torch.tensor(s).to(DEVICE)

BATCHES = [ioi_batch(t, 16 if FAST_DEV_RUN else 64) for t in TEMPLATES]

def logit_diff(logits, io, s):
    last = logits[:, -1]; return (last.gather(1, io[:, None]) - last.gather(1, s[:, None])).mean().item()

with torch.no_grad():
    LD = [(logit_diff(model(c).logits, io, s), logit_diff(model(x).logits, io, s)) for c, x, io, s in BATCHES]
print("LD clean / corrupted по шаблонам:", [(round(a, 2), round(b, 2)) for a, b in LD])

In [ ]:
def patched_ld(batch, hook_module, hook_fn, pre=False):
    clean, corr, io, s = batch
    h = hook_module.register_forward_pre_hook(hook_fn) if pre else hook_module.register_forward_hook(hook_fn)
    try:
        with torch.no_grad(): return logit_diff(model(corr).logits, io, s)
    finally: h.remove()

def recovery(ld, clean_ld, corr_ld): return (ld - corr_ld) / (clean_ld - corr_ld)

# TODO: ваш код ((а) residual patching L×T для первого шаблона; (б) patching z всех голов, усреднение по шаблонам)

## 4. Абляция найденных голов

Mean-ablation: заменить \(z_{l,h}\) на его среднее по всем промптам и позициям чистого датасета. Сравните падение LD на чистых промптах при абляции \(k \in \{3, 6, 10\}\) голов с наибольшим положительным эффектом из п. 3 и \(k\) случайных голов (10 случайных наборов).

In [ ]:
# TODO: ваш код (функция ablate(heads) и таблица LD для найденных и случайных голов)

## 5. Дополнительно: induction heads по ходу обучения (Pythia)

Pythia-70m опубликована с промежуточными чекпойнтами (`revision="step1000"` и т. д.). Для чекпойнтов 0, 500, 1000, 2000, 4000, 8000, 143000 посчитайте максимальный induction score и разность loss первой и второй половин повторённой последовательности. Выполняется только при `FAST_DEV_RUN = False` (около 1 ГБ загрузок).

In [ ]:
if not FAST_DEV_RUN:
    # TODO: ваш код (цикл по revision, max induction score и Δloss, график)
    pass
else:
    print("Пропущено в FAST_DEV_RUN.")

## Вопросы

1. Почему logit lens плохо работает на первых слоях GPT-2 и что исправляет tuned lens?
2. Как induction head реализуется композицией двух голов (K-композиция с previous token head)? Почему для этого нужны минимум два слоя?
3. Почему при patching важен выбор испорченного распределения (ABC vs замена имён местами)? Как это влияет на интерпретацию?
4. Чем mean-ablation лучше zero-ablation? Что такое backup name mover heads и как они проявляются в п. 4?
5. Какие головы IOI-цепи (Wang et al., 2022) вы нашли: name movers, S-inhibition, duplicate token, induction? Подтвердите паттернами внимания хотя бы для одной из них.